In [ ]:
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.graph import START, END, StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver, InMemorySaver

In [ ]:
class MessageState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]



llm = ChatGoogleGenerativeAI(model="gemini-flash-lite-latest")

In [ ]:
def chat_node(state: MessageState):

    # take user query from state
    messages = state['messages']

    # send query to llm
    response = llm.invoke(messages)

    # store response to state
    return {"messages": [response]}

In [ ]:
checkpointer = MemorySaver()        # where to store state of graph when end
graph = StateGraph(MessageState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer=checkpointer)

In [12]:
thread_id = '1'

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "exit", "bye"]:
        break

    config = {"configurable": {"thread_id": thread_id}}
    print("\nYou:", user_input)
    print("AI: ", end="")
    for message_chunk, metadata in chatbot.stream({"messages": HumanMessage(content=user_input)}, config=config, stream_mode="messages"):
        if message_chunk.text:
            print(message_chunk.text, end="", flush=True)



You: can you list our oversation history?
AI: Here is the history of our conversation so far, Ahmad:

1. **You:** hi
   **Me:** Hi there! How can I help you today?

2. **You:** hi
   **Me:** Hello! How's it going?

3. **You:** write 200 words paragraph on pakistan airforce
   **Me:** *(Wrote a 200-word paragraph about the PAF)*

4. **You:** write 200 words paragraph on pakistan airforce
   **Me:** *(Wrote another 200-word paragraph about the PAF)*

5. **You:** write 1000 words paragraph on pakistan airforce
   **Me:** *(Wrote a 1,000-word essay/paragraph about the PAF)*

6. **You:** can you tell me my name?
   **Me:** I don't know your name! ... What is your name?

7. **You:** my name is ahmad
   **Me:** Nice to meet you, Ahmad! How can I help you today?

8. **You:** what is your name?
   **Me:** I don't have a personal name... You can just call me AI or assistant! ...

9. **You:** can you tell me my name
   **Me:** Your name is Ahmad!

10. **You:** can you list our oversation history

### See History

In [ ]:
chatbot.get_state(config=config)        # last saved state

In [ ]:
list(chatbot.get_state_history(config=config))      # every saved state